# Consolidated Storage Efficiency & Benchmark Pipeline
This notebook demonstrates a unified, production-grade storage comparison experiment between **Parquet** and **ORC** file formats under Snappy compression. It performs the following steps in a single pipeline:
1. Clones the sample dataset and decompresses it.
2. Sets up environmental variables and requirements (PySpark, PyArrow).
3. Initializes a local Spark Session.
4. Reads, converts, and evaluates directory footprints dynamically.
5. Performs optimal format decisions based on calculated storage efficiency.

In [14]:
# =====================================================================
# STEP 0: PRE-FLIGHT VERIFICATION OF TPC-DS REPOSITORY BRANCHES
#
# Objective:
# Query the remote GitHub repository to verify that the target scale factors
# (sf1, sf2, sf5) are available as distinct branches prior to cloning.
# =====================================================================

!git ls-remote --heads https://github.com/alamb/tpcds-data.git

105fcb45994f40065c6864c24aa883dd34f4c1a0	refs/heads/main
b41f5010514222d94e50127762868aec11f6e4c5	refs/heads/sf1
eb40d5fe5ee2731bcb054bcdf2f63500a8b5c8bf	refs/heads/sf2
be462d01193c748652d8b21efff1910faad4edb0	refs/heads/sf5


In [16]:
# =====================================================================
# MASTER UNIFIED TPC-DS STORAGE EFFICIENCY PIPELINE - SF1
# =====================================================================

import os
import platform
import subprocess
import pandas as pd

# --- Step 1: Clone TPC-DS SF1 Dataset & Decompress Data ---
# This section clones the light scale factor 1 (SF1) branch of the TPC-DS data repository.
# The files are downloaded as split compressed volumes and then reconstructed using cat, bzip2, and tar.
print("=== [1/6] Cloning & Extracting Datasets ===")
if not os.path.exists("/content/tpcds_sf1/data"):
    print("SF1 data directory not found. Initiating git clone & extraction...")
    subprocess.run("rm -rf /content/tpcds_sf1", shell=True, check=True) # clean up potential half-clones
    subprocess.run("git clone --single-branch --branch sf1 --depth 1 https://github.com/alamb/tpcds-data.git /content/tpcds_sf1", shell=True, check=True)
    subprocess.run("cd /content/tpcds_sf1 && cat data.tar.bz2.* | bzip2 -d | tar -xf -", shell=True, check=True)
    print("SF1 dataset cloned and decompressed successfully!")
else:
    print("SF1 dataset already extracted and verified.")

# --- Step 2: System and Dependency Setup ---
# Ensure PySpark and PyArrow are installed for execution.
# This environment check audits library compatibility across dependencies.
print("\n=== [2/6] Configuring Dependencies ===")
subprocess.run("pip install -q pyspark pyarrow", shell=True, check=True)

import pyarrow
import pyspark
from pyspark.sql import SparkSession

print(f"System Python Version:   {platform.python_version()}")
print(f"Pandas Version:          {pd.__version__}")
print(f"PySpark Version:         {pyspark.__version__}")
print(f"PyArrow Version:         {pyarrow.__version__}")

# --- Step 3: Spark Session Initialization ---
# Configures spark locally using 5 shuffle partitions to optimize localized processing memory.
print("\n=== [3/6] Starting Spark Session ===")
spark = SparkSession.builder \
    .appName("Unified_Parquet_ORC_Experiment") \
    .config("spark.sql.shuffle.partitions", "5") \
    .getOrCreate()
print("Spark Session is active and running!")

# --- Step 4: Utility Size Calculator ---
def calculate_directory_size(dir_path):
    """Recursively walks a folder structure to sum the sizes of all files in bytes.
    This is needed because Spark writes parquet/orc tables as partitioned directories containing metadata."""
    total_bytes = 0
    for root, _, files in os.walk(dir_path):
        for file in files:
            file_path = os.path.join(root, file)
            if not os.path.islink(file_path):
                total_bytes += os.path.getsize(file_path)
    return total_bytes

# --- Step 5: Benchmark Loop & Format Conversions ---
# We iterate over the key dimension and fact tables to benchmark performance and format storage efficiency.
print("\n=== [4/6] Executing Format Conversion Benchmark ===")
target_datasets = [
    "customer.dat",
    "item.dat",
    "store.dat",
    "date_dim.dat",
    "store_sales.dat"
]

benchmark_results = []

for file_name in target_datasets:
    raw_file_path = f"/content/tpcds_sf1/data/{file_name}"
    print(f"Converting & Measuring {file_name}...")

    # Read pipe-delimited file using Spark SQL engine
    df_spark_dataset = spark.read.csv(raw_file_path, sep="|", inferSchema=True)

    parquet_output_path = f"/content/{file_name}_parquet"
    orc_output_path = f"/content/{file_name}_orc"

    # Convert and write dataset formats using Snappy compression scheme
    df_spark_dataset.write.mode("overwrite") \
        .option("compression", "snappy") \
        .parquet(parquet_output_path)

    df_spark_dataset.write.mode("overwrite") \
        .option("compression", "snappy") \
        .orc(orc_output_path)

    # Capture physical file footprints
    raw_bytes = os.path.getsize(raw_file_path)
    parquet_bytes = calculate_directory_size(parquet_output_path)
    orc_bytes = calculate_directory_size(orc_output_path)

    # Determine ratio relative to raw text size
    parquet_ratio = parquet_bytes / raw_bytes
    orc_ratio = orc_bytes / raw_bytes

    # Metrics recorded in MegaBytes (MB) for easier reading
    benchmark_results.append({
        "File": file_name,
        "Raw_Size_MB": round(raw_bytes / (1024 * 1024), 2),
        "Parquet_Size_MB": round(parquet_bytes / (1024 * 1024), 2),
        "ORC_Size_MB": round(orc_bytes / (1024 * 1024), 2),
        "Parquet_Compression_Ratio": round(parquet_ratio, 3),
        "ORC_Compression_Ratio": round(orc_ratio, 3)
    })

# --- Step 6: Format Decision Evaluation ---
# Evaluate and tag the optimal file format based on the smaller relative footprint.
print("\n=== [5/6] Generating Results Summary ===")
comparison_df = pd.DataFrame(benchmark_results)
comparison_df["Optimal_Format"] = comparison_df.apply(
    lambda row: "Parquet" if row["Parquet_Compression_Ratio"] < row["ORC_Compression_Ratio"] else "ORC",
    axis=1
)

print("\n=== [6/6] Unified Storage Efficiency Experiment Results ===")
display(comparison_df)

=== [1/6] Cloning & Extracting Datasets ===
SF1 dataset already extracted and verified.

=== [2/6] Configuring Dependencies ===
System Python Version:   3.13.18
Pandas Version:          2.2.3
PySpark Version:         4.0.5
PyArrow Version:         23.0.1

=== [3/6] Starting Spark Session ===
Spark Session is active and running!

=== [4/6] Executing Format Conversion Benchmark ===
Converting & Measuring customer.dat...
Converting & Measuring item.dat...
Converting & Measuring store.dat...
Converting & Measuring date_dim.dat...
Converting & Measuring store_sales.dat...

=== [5/6] Generating Results Summary ===

=== [6/6] Unified Storage Efficiency Experiment Results ===


,File,Raw_Size_MB,Parquet_Size_MB,ORC_Size_MB,Parquet_Compression_Ratio,ORC_Compression_Ratio,Optimal_Format
0,customer.dat,12.60,5.34,5.16,0.424,0.409,ORC
1,item.dat,4.82,1.72,1.64,0.356,0.341,ORC
2,store.dat,0.00,0.01,0.01,3.313,1.725,ORC
3,date_dim.dat,9.84,1.80,0.48,0.183,0.049,ORC
4,store_sales.dat,370.45,140.81,164.42,0.380,0.444,Parquet


In [17]:
# =====================================================================
# MASTER UNIFIED TPC-DS STORAGE EFFICIENCY PIPELINE - SF2
# =====================================================================

import os
import platform
import subprocess
import pandas as pd

# --- Step 1: Clone TPC-DS SF2 Dataset & Decompress Data ---
# This section downloads the Scale Factor 2 (SF2) dataset using the dedicated sf2 git branch.
print("=== [1/6] Cloning & Extracting Datasets ===")
required_file = "/content/tpcds_sf2/data/customer.dat"
if not os.path.exists(required_file):
    print("SF2 dataset files not found or incomplete. Initiating clean clone and extraction...")
    subprocess.run("rm -rf /content/tpcds_sf2", shell=True, check=True)
    subprocess.run("git clone --single-branch --branch sf2 --depth 1 https://github.com/alamb/tpcds-data.git /content/tpcds_sf2", shell=True, check=True)
    subprocess.run("cd /content/tpcds_sf2 && cat data.tar.bz2.* | bzip2 -d | tar -xf -", shell=True, check=True)
    print("SF2 dataset cloned and decompressed successfully!")
else:
    print("SF2 dataset already extracted and verified.")

# --- Step 2: System and Dependency Setup ---
print("\n=== [2/6] Configuring Dependencies ===")
subprocess.run("pip install -q pyspark pyarrow", shell=True, check=True)

import pyarrow
import pyspark
from pyspark.sql import SparkSession

# Check software runtime details
print(f"System Python Version:   {platform.python_version()}")
print(f"Pandas Version:          {pd.__version__}")
print(f"PySpark Version:         {pyspark.__version__}")
print(f"PyArrow Version:         {pyarrow.__version__}")

# --- Step 3: Spark Session Initialization ---
print("\n=== [3/6] Starting Spark Session ===")
spark = SparkSession.builder 	.appName("Unified_Parquet_ORC_Experiment") 	.config("spark.sql.shuffle.partitions", "5") 	.getOrCreate()
print("Spark Session is active and running!")

# --- Step 4: Utility Size Calculator ---
def calculate_directory_size(dir_path):
    """Calculates physical size of a multi-part directory created by Spark partition tasks."""
    total_bytes = 0
    for root, _, files in os.walk(dir_path):
        for file in files:
            file_path = os.path.join(root, file)
            if not os.path.islink(file_path):
                total_bytes += os.path.getsize(file_path)
    return total_bytes

# --- Step 5: Benchmark Loop & Format Conversions ---
print("\n=== [4/6] Executing Format Conversion Benchmark ===")
target_datasets = [
    "customer.dat",
    "item.dat",
    "store.dat",
    "date_dim.dat",
    "store_sales.dat"
]

benchmark_results = []

for file_name in target_datasets:
    raw_file_path = f"/content/tpcds_sf2/data/{file_name}"
    print(f"Converting & Measuring {file_name}...")

    # Read Raw Structured CSV File
    df_spark_dataset = spark.read.csv(raw_file_path, sep="|", inferSchema=True)

    parquet_output_path = f"/content/{file_name}_parquet"
    orc_output_path = f"/content/{file_name}_orc"

    # Save in target binary columnar layouts
    df_spark_dataset.write.mode("overwrite") \
        .option("compression", "snappy") \
        .parquet(parquet_output_path)

    df_spark_dataset.write.mode("overwrite") \
        .option("compression", "snappy") \
        .orc(orc_output_path)

    # Calculate size footprint details
    raw_bytes = os.path.getsize(raw_file_path)
    parquet_bytes = calculate_directory_size(parquet_output_path)
    orc_bytes = calculate_directory_size(orc_output_path)

    parquet_ratio = parquet_bytes / raw_bytes
    orc_ratio = orc_bytes / raw_bytes

    # Format and store metrics metadata
    benchmark_results.append({
        "File": file_name,
        "Raw_Size_MB": round(raw_bytes / (1024 * 1024), 2),
        "Parquet_Size_MB": round(parquet_bytes / (1024 * 1024), 2),
        "ORC_Size_MB": round(orc_bytes / (1024 * 1024), 2),
        "Parquet_Compression_Ratio": round(parquet_ratio, 3),
        "ORC_Compression_Ratio": round(orc_ratio, 3)
    })

# --- Step 6: Format Decision Evaluation ---
print("\n=== [5/6] Generating Results Summary ===")
comparison_df = pd.DataFrame(benchmark_results)
comparison_df["Optimal_Format"] = comparison_df.apply(
    lambda row: "Parquet" if row["Parquet_Compression_Ratio"] < row["ORC_Compression_Ratio"] else "ORC",
    axis=1
)

print("\n=== [6/6] Unified Storage Efficiency Experiment Results ===")
display(comparison_df)

=== [1/6] Cloning & Extracting Datasets ===
SF2 dataset already extracted and verified.

=== [2/6] Configuring Dependencies ===
System Python Version:   3.13.18
Pandas Version:          2.2.3
PySpark Version:         4.0.5
PyArrow Version:         23.0.1

=== [3/6] Starting Spark Session ===
Spark Session is active and running!

=== [4/6] Executing Format Conversion Benchmark ===
Converting & Measuring customer.dat...
Converting & Measuring item.dat...
Converting & Measuring store.dat...
Converting & Measuring date_dim.dat...
Converting & Measuring store_sales.dat...

=== [5/6] Generating Results Summary ===

=== [6/6] Unified Storage Efficiency Experiment Results ===


,File,Raw_Size_MB,Parquet_Size_MB,ORC_Size_MB,Parquet_Compression_Ratio,ORC_Compression_Ratio,Optimal_Format
0,customer.dat,18.20,7.59,7.39,0.417,0.406,ORC
1,item.dat,6.98,2.47,2.37,0.354,0.340,ORC
2,store.dat,0.01,0.01,0.01,1.993,1.120,ORC
3,date_dim.dat,9.84,1.80,0.48,0.183,0.049,ORC
4,store_sales.dat,747.71,286.10,329.84,0.383,0.441,Parquet


In [18]:
# =====================================================================
# MASTER UNIFIED TPC-DS STORAGE EFFICIENCY PIPELINE - SF5
# =====================================================================

import os
import platform
import subprocess
import pandas as pd

# --- Step 1: Clone TPC-DS SF5 Dataset & Decompress Data ---
# This section sets up Scale Factor 5 (SF5), representing a larger database environment footprint (~5GB raw)
print("=== [1/6] Cloning & Extracting Datasets ===")
required_file = "/content/tpcds_sf5/data/customer.dat"
if not os.path.exists(required_file):
    print("SF5 dataset files not found or incomplete. Initiating clean clone and extraction...")
    subprocess.run("rm -rf /content/tpcds_sf5", shell=True, check=True)
    subprocess.run("git clone --single-branch --branch sf5 --depth 1 https://github.com/alamb/tpcds-data.git /content/tpcds_sf5", shell=True, check=True)
    subprocess.run("cd /content/tpcds_sf5 && cat data.tar.bz2.* | bzip2 -d | tar -xf -", shell=True, check=True)
    print("SF5 dataset cloned and decompressed successfully!")
else:
    print("SF5 dataset already extracted and verified.")

# --- Step 2: System and Dependency Setup ---
print("\n=== [2/6] Configuring Dependencies ===")
subprocess.run("pip install -q pyspark pyarrow", shell=True, check=True)

import pyarrow
import pyspark
from pyspark.sql import SparkSession

print(f"System Python Version:   {platform.python_version()}")
print(f"Pandas Version:          {pd.__version__}")
print(f"PySpark Version:         {pyspark.__version__}")
print(f"PyArrow Version:         {pyarrow.__version__}")

# --- Step 3: Spark Session Initialization ---
print("\n=== [3/6] Starting Spark Session ===")
spark = SparkSession.builder 	.appName("Unified_Parquet_ORC_Experiment") 	.config("spark.sql.shuffle.partitions", "5") 	.getOrCreate()
print("Spark Session is active and running!")

# --- Step 4: Utility Size Calculator ---
def calculate_directory_size(dir_path):
    """Returns consolidated sizes in bytes of a partition repository directory."""
    total_bytes = 0
    for root, _, files in os.walk(dir_path):
        for file in files:
            file_path = os.path.join(root, file)
            if not os.path.islink(file_path):
                total_bytes += os.path.getsize(file_path)
    return total_bytes

# --- Step 5: Benchmark Loop & Format Conversions ---
print("\n=== [4/6] Executing Format Conversion Benchmark ===")
target_datasets = [
    "customer.dat",
    "item.dat",
    "store.dat",
    "date_dim.dat",
    "store_sales.dat"
]

benchmark_results = []

for file_name in target_datasets:
    raw_file_path = f"/content/tpcds_sf5/data/{file_name}"
    print(f"Converting & Measuring {file_name}...")

    # Read Raw Structured CSV File
    df_spark_dataset = spark.read.csv(raw_file_path, sep="|", inferSchema=True)

    parquet_output_path = f"/content/{file_name}_parquet"
    orc_output_path = f"/content/{file_name}_orc"

    # Export file streams to target file configurations
    df_spark_dataset.write.mode("overwrite") \
        .option("compression", "snappy") \
        .parquet(parquet_output_path)

    df_spark_dataset.write.mode("overwrite") \
        .option("compression", "snappy") \
        .orc(orc_output_path)

    raw_bytes = os.path.getsize(raw_file_path)
    parquet_bytes = calculate_directory_size(parquet_output_path)
    orc_bytes = calculate_directory_size(orc_output_path)

    parquet_ratio = parquet_bytes / raw_bytes
    orc_ratio = orc_bytes / raw_bytes

    benchmark_results.append({
        "File": file_name,
        "Raw_Size_MB": round(raw_bytes / (1024 * 1024), 2),
        "Parquet_Size_MB": round(parquet_bytes / (1024 * 1024), 2),
        "ORC_Size_MB": round(orc_bytes / (1024 * 1024), 2),
        "Parquet_Compression_Ratio": round(parquet_ratio, 3),
        "ORC_Compression_Ratio": round(orc_ratio, 3)
    })

# --- Step 6: Format Decision Evaluation ---
print("\n=== [5/6] Generating Results Summary ===")
comparison_df = pd.DataFrame(benchmark_results)
comparison_df["Optimal_Format"] = comparison_df.apply(
    lambda row: "Parquet" if row["Parquet_Compression_Ratio"] < row["ORC_Compression_Ratio"] else "ORC",
    axis=1
)

print("\n=== [6/6] Unified Storage Efficiency Experiment Results ===")
display(comparison_df)

=== [1/6] Cloning & Extracting Datasets ===
SF5 dataset already extracted and verified.

=== [2/6] Configuring Dependencies ===
System Python Version:   3.13.18
Pandas Version:          2.2.3
PySpark Version:         4.0.5
PyArrow Version:         23.0.1

=== [3/6] Starting Spark Session ===
Spark Session is active and running!

=== [4/6] Executing Format Conversion Benchmark ===
Converting & Measuring customer.dat...
Converting & Measuring item.dat...
Converting & Measuring store.dat...
Converting & Measuring date_dim.dat...
Converting & Measuring store_sales.dat...

=== [5/6] Generating Results Summary ===

=== [6/6] Unified Storage Efficiency Experiment Results ===


,File,Raw_Size_MB,Parquet_Size_MB,ORC_Size_MB,Parquet_Compression_Ratio,ORC_Compression_Ratio,Optimal_Format
0,customer.dat,35.19,14.41,14.13,0.409,0.401,ORC
1,item.dat,14.50,5.07,4.87,0.350,0.336,ORC
2,store.dat,0.01,0.01,0.01,1.081,0.699,ORC
3,date_dim.dat,9.84,1.80,0.48,0.183,0.049,ORC
4,store_sales.dat,1889.46,716.46,841.27,0.379,0.445,Parquet


In [19]:
# =====================================================================
# EXPERIMENT: READ & WRITE PERFORMANCE COMPARISON for SF1
#
# Objective:
# Compare performance benchmarks for parquet versus orc file formats under SF1
# 1. Write times (seconds)
# 2. Sequential/Aggregated read times (seconds)
# =====================================================================

import time
import pandas as pd

files = [
    "customer.dat",
    "item.dat",
    "store.dat",
    "date_dim.dat",
    "store_sales.dat"
]

performance_results = []

# Iterate through each dataset file to process write and read timings
for file_name in files:

    print(f"Processing {file_name} ...")
    file_path = f"/content/tpcds_sf1/data/{file_name}"

    # 1. Load source raw database text structure in Spark
    df = spark.read.csv(
        file_path,
        sep="|",
        inferSchema=True
    )

    # 2. Capture and record Parquet file writing execution time
    parquet_path = f"/content/{file_name}_parquet_perf"
    start = time.time()
    df.write.mode("overwrite") \
        .option("compression", "snappy") \
        .parquet(parquet_path)
    parquet_write_time = time.time() - start

    # 3. Capture and record ORC file writing execution time
    orc_path = f"/content/{file_name}_orc_perf"
    start = time.time()
    df.write.mode("overwrite") \
        .option("compression", "snappy") \
        .orc(orc_path)
    orc_write_time = time.time() - start

    # 4. Measure read performance by loading data and forcing partition scan with count()
    start = time.time()
    spark.read.parquet(parquet_path).count()
    parquet_read_time = time.time() - start

    # 5. Measure ORC read execution duration
    start = time.time()
    spark.read.orc(orc_path).count()
    orc_read_time = time.time() - start

    # Compile timing results array
    performance_results.append({
        "File": file_name,
        "Parquet_Write_Sec": round(parquet_write_time, 2),
        "ORC_Write_Sec": round(orc_write_time, 2),
        "Parquet_Read_Sec": round(parquet_read_time, 2),
        "ORC_Read_Sec": round(orc_read_time, 2)
    })

# Output results as a pandas DataFrame comparison dataset
performance_df = pd.DataFrame(performance_results)
display(performance_df)

Processing customer.dat ...
Processing item.dat ...
Processing store.dat ...
Processing date_dim.dat ...
Processing store_sales.dat ...


,File,Parquet_Write_Sec,ORC_Write_Sec,Parquet_Read_Sec,ORC_Read_Sec
0,customer.dat,3.40,1.73,0.80,0.49
1,item.dat,1.24,0.77,0.30,0.27
2,store.dat,0.17,0.17,0.29,0.22
3,date_dim.dat,1.34,1.17,0.29,0.17
4,store_sales.dat,27.68,23.62,0.33,0.29


In [20]:
# =====================================================================
# EXPERIMENT: READ & WRITE PERFORMANCE COMPARISON for SF2
#
# Objective:
# Compare performance benchmarks for Parquet versus ORC file formats under SF2
# 1. Write times (seconds)
# 2. Sequential/Aggregated read times (seconds)
# =====================================================================

import time
import pandas as pd

files = [
    "customer.dat",
    "item.dat",
    "store.dat",
    "date_dim.dat",
    "store_sales.dat"
]

performance_results = []

# Iterate through each dataset file to process write and read timings
for file_name in files:
    print(f"Processing {file_name} ...")
    file_path = f"/content/tpcds_sf2/data/{file_name}"

    # 1. Load raw file structure into Spark
    df = spark.read.csv(
        file_path,
        sep="|",
        inferSchema=True
    )

    # 2. Capture and record Parquet file writing execution duration
    parquet_path = f"/content/{file_name}_parquet_perf"
    start = time.time()
    df.write.mode("overwrite") \
        .option("compression", "snappy") \
        .parquet(parquet_path)
    parquet_write_time = time.time() - start

    # 3. Capture and record ORC file writing execution duration
    orc_path = f"/content/{file_name}_orc_perf"
    start = time.time()
    df.write.mode("overwrite") \
        .option("compression", "snappy") \
        .orc(orc_path)
    orc_write_time = time.time() - start

    # 4. Measure read performance by triggering a full action count
    start = time.time()
    spark.read.parquet(parquet_path).count()
    parquet_read_time = time.time() - start

    # 5. Measure ORC read performance by triggering a full action count
    start = time.time()
    spark.read.orc(orc_path).count()
    orc_read_time = time.time() - start

    # Save metrics results
    performance_results.append({
        "File": file_name,
        "Parquet_Write_Sec": round(parquet_write_time, 2),
        "ORC_Write_Sec": round(orc_write_time, 2),
        "Parquet_Read_Sec": round(parquet_read_time, 2),
        "ORC_Read_Sec": round(orc_read_time, 2)
    })

# Present unified metrics dashboard
performance_df = pd.DataFrame(performance_results)
display(performance_df)

Processing customer.dat ...
Processing item.dat ...
Processing store.dat ...
Processing date_dim.dat ...
Processing store_sales.dat ...


,File,Parquet_Write_Sec,ORC_Write_Sec,Parquet_Read_Sec,ORC_Read_Sec
0,customer.dat,1.28,1.37,0.30,0.23
1,item.dat,0.55,0.79,0.19,0.20
2,store.dat,0.17,0.18,0.25,0.26
3,date_dim.dat,1.66,1.31,0.25,0.16
4,store_sales.dat,53.33,44.87,0.59,0.19


In [21]:
# =====================================================================
# EXPERIMENT: READ & WRITE PERFORMANCE COMPARISON for SF5
#
# Objective:
# Compare Parquet vs ORC performance for SF5 on key dimensions:
# 1. Write Duration (seconds)
# 2. Read / Scan Duration (seconds)
#
# Files Tested:
# - customer.dat
# - item.dat
# - store.dat
# - date_dim.dat
# - store_sales.dat
# =====================================================================

import time
import pandas as pd

# Define the target dataset files from TPC-DS schema
files = [
    "customer.dat",
    "item.dat",
    "store.dat",
    "date_dim.dat",
    "store_sales.dat"
]

# Container to hold computed performance metrics
sf5_performance_results = []

# --------------------------------------------------
# Main Benchmarking Loop
# --------------------------------------------------
for file_name in files:
    print(f"Processing SF5 {file_name} ...")
    file_path = f"/content/tpcds_sf5/data/{file_name}"

    # 1. Load CSV source file using Spark with schema inference
    df_sf5 = spark.read.csv(
        file_path,
        sep="|",
        inferSchema=True
    )

    # 2. Benchmark Parquet Write Performance with Snappy Compression
    parquet_path_sf5 = f"/content/{file_name}_parquet_perf_sf5"
    start = time.time()
    df_sf5.write.mode("overwrite") \
        .option("compression", "snappy") \
        .parquet(parquet_path_sf5)
    parquet_write_time = time.time() - start

    # 3. Benchmark ORC Write Performance with Snappy Compression
    orc_path_sf5 = f"/content/{file_name}_orc_perf_sf5"
    start = time.time()
    df_sf5.write.mode("overwrite") \
        .option("compression", "snappy") \
        .orc(orc_path_sf5)
    orc_write_time = time.time() - start

    # 4. Benchmark Parquet Read (Scan & Action) Performance via Count
    start = time.time()
    spark.read.parquet(parquet_path_sf5).count()
    parquet_read_time = time.time() - start

    # 5. Benchmark ORC Read (Scan & Action) Performance via Count
    start = time.time()
    spark.read.orc(orc_path_sf5).count()
    orc_read_time = time.time() - start

    # Append calculated times rounded to 2 decimal places
    sf5_performance_results.append({
        "File": file_name,
        "Parquet_Write_Sec": round(parquet_write_time, 2),
        "ORC_Write_Sec": round(orc_write_time, 2),
        "Parquet_Read_Sec": round(parquet_read_time, 2),
        "ORC_Read_Sec": round(orc_read_time, 2)
    })

# --------------------------------------------------
# Display Performance Comparison Results DataFrame
# --------------------------------------------------
sf5_performance_df = pd.DataFrame(sf5_performance_results)
display(sf5_performance_df)

Processing SF5 customer.dat ...
Processing SF5 item.dat ...
Processing SF5 store.dat ...
Processing SF5 date_dim.dat ...
Processing SF5 store_sales.dat ...


,File,Parquet_Write_Sec,ORC_Write_Sec,Parquet_Read_Sec,ORC_Read_Sec
0,customer.dat,2.18,2.64,0.18,0.16
1,item.dat,1.03,1.32,0.30,0.11
2,store.dat,0.15,0.22,0.18,0.11
3,date_dim.dat,1.48,1.95,0.35,0.14
4,store_sales.dat,133.53,112.25,0.76,0.50


In [22]:
#Machine information

!lscpu | grep "Model name"; nproc; free -h; df -h /content

Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
2
               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.2Gi       6.5Gi       3.1Mi       4.3Gi        10Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   18G   91G  16% /
